# 02 Two Option Filter Batch (пакетный двухопционный фильтр)

Этот notebook (ноутбук) легкий: вся тяжелая логика лежит в `two_option_filter_batch.py (файле Python)`.

Он читает `core paths (траектории базовых стратегий)` из результатов первого шага и обрабатывает их пакетами по `150 strategies (150 стратегий)`. Каждый batch (пакет) запускается отдельным Python process (процессом Python), сохраняет результаты на D и завершает работу.

## Что проверяется

Для каждой `core strategy (базовой стратегии)` на `train (обучающем периоде)` прогоняются четыре конструкции:

- `call/call (колл/колл)`;
- `call/put (колл/пут)`;
- `put/call (пут/колл)`;
- `put/put (пут/пут)`.

Сетка параметров включает `T1/T2 (экспирации первого и второго слоя)`, `std spans (окна волатильности)` и `K shifts (сдвиги страйка)`. В полный файл batch (пакета) сохраняются все метрики сетки, а для каждой стратегии отдельно сохраняется только best path (лучшая траектория), чтобы не забивать диск и память.

In [1]:
from pathlib import Path
import subprocess
import sys

WORK_ROOT = Path(r"./workspace")
SCRIPT = WORK_ROOT / "two_option_filter_batch.py"

cmd = [sys.executable, str(SCRIPT), "--mode", "summary"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Total core strategies (всего базовых стратегий): 1636
Batch size (размер пакета): 150
Batch ids (номера пакетов): 0 .. 10
family
pair_spread_mean_reversion         540
pair_spread_momentum               540
time_series_momentum               120
time_series_reversal               120
moving_average_crossover            72
channel_breakout                    72
channel_reversal                    72
sector_cross_sectional_reversal     25
sector_cross_sectional_momentum     25
sector_trend_portfolio              25
sector_reversal_portfolio           25




## Запуск одного batch (пакета)

Меняй только `BATCH_ID (номер пакета)`. При `BATCH_SIZE (размере пакета)` 150 нужно запустить номера от `0` до `10`.

In [3]:
for i in range(1, 11):
    BATCH_ID = i
    BATCH_SIZE = 150
    
    cmd = [
        sys.executable,
        str(SCRIPT),
        "--mode", "batch",
        "--batch-id", str(BATCH_ID),
        "--batch-size", str(BATCH_SIZE),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)

{
  "batch_label": "0001",
  "start_spec_id": 150,
  "end_spec_id_exclusive": 300,
  "n_catalog_total": 1636,
  "n_strategies": 150,
  "n_grid_rows": 1200,
  "n_best_rows": 150,
  "n_best_paths_saved": 150,
  "output_dir": "D:\\Filter_2_option_v2\\02_two_option_filter_output"
}


{
  "batch_label": "0002",
  "start_spec_id": 300,
  "end_spec_id_exclusive": 450,
  "n_catalog_total": 1636,
  "n_strategies": 150,
  "n_grid_rows": 1200,
  "n_best_rows": 150,
  "n_best_paths_saved": 150,
  "output_dir": "D:\\Filter_2_option_v2\\02_two_option_filter_output"
}


{
  "batch_label": "0003",
  "start_spec_id": 450,
  "end_spec_id_exclusive": 600,
  "n_catalog_total": 1636,
  "n_strategies": 150,
  "n_grid_rows": 1200,
  "n_best_rows": 150,
  "n_best_paths_saved": 150,
  "output_dir": "D:\\Filter_2_option_v2\\02_two_option_filter_output"
}


{
  "batch_label": "0004",
  "start_spec_id": 600,
  "end_spec_id_exclusive": 750,
  "n_catalog_total": 1636,
  "n_strategies": 150,
  "n_grid_rows": 1200,
 

## Audit (аудит) и combine (объединение)

Audit (аудит) показывает пропущенные диапазоны. Combine (объединение) собирает итоговую таблицу лучших двухопционных вариантов.

In [2]:
cmd = [sys.executable, str(SCRIPT), "--mode", "audit"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

{
  "catalog_rows": 1636,
  "processed_rows": 1636,
  "processed_unique": 1636,
  "missing_count": 0,
  "duplicate_count": 0,
  "missing_ranges": [],
  "duplicate_head": []
}




In [3]:
cmd = [sys.executable, str(SCRIPT), "--mode", "combine"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

{
  "n_grid_batch_files": 11,
  "n_best_batch_files": 11,
  "n_unique_strategies": 1636,
  "n_best_paths": 1636,
  "output_dir": "D:\\Filter_2_option_v2\\02_two_option_filter_output"
}




## Результаты

Все results (результаты) сохраняются на диск D:

`D:\Filter_2_option_v2\02_two_option_filter_output`

Главные файлы:

- `batches/grid_batch_XXXX.csv` - вся сетка фильтров для одного batch (пакета);
- `batches/best_batch_XXXX.csv` - лучший фильтр для каждой стратегии в batch (пакете);
- `03_two_option_best_by_strategy.csv` - итоговая таблица после combine (объединения);
- `04_two_option_best_paths.csv` - пути к best paths (лучшим траекториям);
- `05_manifest.json` - паспорт итоговой сборки.